# Scan Geo 04 — Valdivia: Localización de Cuerpos de Agua Más Cercanos

Este notebook calcula la **distancia euclidiana directa en metros** desde cada uno de los **126 puntos de medición** de Valdivia hasta los cuerpos de agua más cercanos (ríos, esteros, lagunas y masas lacustres) utilizando la cartografía hidrológica en formato Shapefile.

### Archivos de Entrada:
- **Red Hidrográfica (Ríos/Drenes):** `01_INPUT/Red_Hidrografica/Red_Hidrografica.shp`
- **Masas Lacustres (Lagos/Lagunas):** `01_INPUT/Masas_Lacustres/masas_lacustres.shp`
- **Puntos de Medición (126 Sensores):** `/home/oscar/Escritorio/x_pruebas/osm/scan_geo04_valdivia_liu/salidas_scan/req4_puntos_medicion_morfologia.csv`

---


In [1]:
import os
import sys
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from shapely.geometry import Point, LineString, box
import folium
from folium import FeatureGroup, LayerControl

# --- VARIABLE MAESTRA ---
NOMBRE_CIUDAD = "padrelascasas_abr2014"       # Etiqueta de la ciudad (coincide con los pasos 0 a 3)
BUFFER_BUSQUEDA_M = 5000.0      # Buffer de 15 km alrededor de los sensores para recortar ríos/lagos

# Rutas locales
MODELO_01 = Path("01_clas")
INPUT_AGUA_DIR = Path("04_agu/00_in")
DATOS_OUT = Path("04_agu")
APP_DIR = Path("04_agu")

# DATOS_DIR = Path("02_datos")
# INPUT_AGUA_DIR = Path("03_input_agua")
# SALIDAS_DIR = Path("salidas_scan")
# APP_DIR = Path("app_interactiva")


RIO_SHP = INPUT_AGUA_DIR / "Red_Hidrografica" / "Red_Hidrografica.shp"
LAGO_SHP = INPUT_AGUA_DIR / "Masas_Lacustres" / "masas_lacustres.shp"
# SENSOR_CSV = Path("/home/oscar/Escritorio/x_pruebas/osm/scan_geo04_valdivia_liu/salidas_scan/req4_puntos_medicion_morfologia.csv")     #Para Valdivia
# SENSOR_CSV = Path("/home/oscar/Escritorio/x_pruebas/aumilab-experimentos/03_kml/Puntos_medicion_data_sant26_test.csv")

print(f"=== ANÁLISIS ESPACIAL DE DISTANCIA A CUERPOS DE AGUA: {NOMBRE_CIUDAD.upper()} ===")
print(f"Carpeta de insumos hidrográficos: {INPUT_AGUA_DIR.resolve()}")

=== ANÁLISIS ESPACIAL DE DISTANCIA A CUERPOS DE AGUA: PADRELASCASAS_ABR2014 ===
Carpeta de insumos hidrográficos: /home/oscar/Escritorio/_Final_15Sep26/scan_completo/04_agu/00_in


In [2]:
# ============================================================
# 2. Cargar Sensores Acústicos
# ============================================================
def deducir_tipo_campana(sensor_id):
    s = str(sensor_id).upper()
    if s.startswith("T"): return "Troncal"
    elif s.startswith("C"): return "Colectora"
    elif s.startswith("S"): return "Servicio"
    elif s.startswith("L"): return "Local"
    elif s.startswith("E") or s.startswith("EM"): return "Estaciones"
    return "Otro"

def normalizar_tipo_campana(row):
    cat = row.get("categoria_oguc")
    if pd.notna(cat) and str(cat).strip().lower() not in ["none", "nan", ""]:
        c = str(cat).strip().capitalize()
        if c.startswith("Troncal"): return "Troncal"
        elif c.startswith("Colect"): return "Colectora"
        elif c.startswith("Servic"): return "Servicio"
        elif c.startswith("Local"): return "Local"
        elif c.startswith("Expres"): return "Expresa"
        elif c.startswith("Estacion"): return "Estaciones"
        return c
    return deducir_tipo_campana(row.get("sensor_id", row.get("id", "")))

# Buscar archivo de mediciones de la ciudad estrictamente según la variable
archivo_med_json = MODELO_01 / f"mediciones_aumilab_{NOMBRE_CIUDAD.lower()}.json"
if not archivo_med_json.exists():
    raise FileNotFoundError(f"No se encontró el archivo de mediciones en: {archivo_med_json}")

with open(archivo_med_json, "r", encoding="utf-8") as f:
    med_raw = json.load(f)
df_sensors = pd.DataFrame(med_raw)

# Estandarizar nombres de columnas
df_sensors["sensor_id"] = df_sensors["id"]
df_sensors["calle_sensor"] = df_sensors.get("calle", "s/n")
df_sensors["latitud"] = df_sensors["lat"].astype(float)
df_sensors["longitud"] = df_sensors["lon"].astype(float)

# Asignar tipo_campana_medicion con prioridad en categoria_oguc
if "tipo_campana_medicion" not in df_sensors.columns or df_sensors["tipo_campana_medicion"].eq("Otro").all():
    df_sensors["tipo_campana_medicion"] = df_sensors.apply(normalizar_tipo_campana, axis=1)

gdf_sensors_4326 = gpd.GeoDataFrame(
    df_sensors,
    geometry=gpd.points_from_xy(df_sensors["longitud"], df_sensors["latitud"]),
    crs=4326
)
gdf_sensors_3857 = gdf_sensors_4326.to_crs(3857)

print(f"Sensores cargados para [{NOMBRE_CIUDAD}]: {len(gdf_sensors_3857)} puntos desde {archivo_med_json.name}")

# Bounding Box alrededor de los sensores para acotar la búsqueda en los shapefiles
minx, miny, maxx, maxy = gdf_sensors_3857.total_bounds
search_box = box(minx - BUFFER_BUSQUEDA_M, miny - BUFFER_BUSQUEDA_M, 
                 maxx + BUFFER_BUSQUEDA_M, maxy + BUFFER_BUSQUEDA_M)
print(f"Área de búsqueda delimitada con buffer de {int(BUFFER_BUSQUEDA_M/1000)} km.")


Sensores cargados para [padrelascasas_abr2014]: 15 puntos desde mediciones_aumilab_padrelascasas_abr2014.json
Área de búsqueda delimitada con buffer de 5 km.


In [3]:
# ============================================================
# 3. Cargar Cartografía Hidrográfica (Ríos y Masas Lacustres)
# ============================================================

print("Cargando Red Hidrográfica (Ríos y Drenes)...", flush=True)
gdf_rios = gpd.read_file(RIO_SHP, bbox=search_box).to_crs(3857)
col_nom_rio = "Nombre" if "Nombre" in gdf_rios.columns else ("nombre" if "nombre" in gdf_rios.columns else None)
col_tipo_rio = "Dren_Tipo" if "Dren_Tipo" in gdf_rios.columns else ("tipo" if "tipo" in gdf_rios.columns else None)

gdf_rios["nombre_clean"] = gdf_rios[col_nom_rio].fillna("Estero Sin Nombre / Dren") if col_nom_rio else "Río / Dren"
gdf_rios["tipo_clean"] = gdf_rios[col_tipo_rio].fillna("Río / Dren") if col_tipo_rio else "Río"
print(f"Ríos en el área de estudio: {len(gdf_rios)} tramos")

# -----

print("Cargando Masas Lacustres (Lagos y Lagunas)...", flush=True)
gdf_lagos = gpd.read_file(LAGO_SHP, bbox=search_box).to_crs(3857)
col_nom_lago = "Nombre" if "Nombre" in gdf_lagos.columns else ("nombre" if "nombre" in gdf_lagos.columns else None)
col_tipo_lago = "Tipo" if "Tipo" in gdf_lagos.columns else ("tipo" if "tipo" in gdf_lagos.columns else None)

gdf_lagos["nombre_clean"] = gdf_lagos[col_nom_lago].fillna("Masa Lacustre / Laguna") if col_nom_lago else "Lago / Laguna"
gdf_lagos["tipo_clean"] = gdf_lagos[col_tipo_lago].fillna("Masa Lacustre") if col_tipo_lago else "Masa Lacustre"
print(f"Masas Lacustres en el área de estudio: {len(gdf_lagos)} polígonos")


Cargando Red Hidrográfica (Ríos y Drenes)...


Ríos en el área de estudio: 65 tramos
Cargando Masas Lacustres (Lagos y Lagunas)...


Masas Lacustres en el área de estudio: 0 polígonos


ERROR 1: PROJ: proj_create_from_database: Open of /home/oscar/miniconda3/envs/geo03-gpu/share/proj failed


In [4]:
# ============================================================
# 4. Cálculo de Distancias Mínimas Euclidianas Proyectadas (sjoin_nearest en EPSG:3857)
# ============================================================

# 1. Distancia más cercana a Ríos
if len(gdf_rios) > 0:
    print("Calculando distancia más cercana a Ríos...", flush=True)
    join_rio = gpd.sjoin_nearest(gdf_sensors_3857, gdf_rios[["nombre_clean", "tipo_clean", "geometry"]], distance_col="dist_rio_m", how="left")
    join_rio = join_rio.groupby(join_rio.index).first()
else:
    print("No se encontraron Ríos/Drenes en el área de estudio.", flush=True)
    join_rio = pd.DataFrame(index=gdf_sensors_3857.index)
    join_rio["nombre_clean"] = "Sin ríos cercanos"
    join_rio["tipo_clean"] = "N/A"
    join_rio["dist_rio_m"] = np.nan
    join_rio["index_right"] = np.nan

# 2. Distancia más cercana a Masas Lacustres
if len(gdf_lagos) > 0:
    print("Calculando distancia más cercana a Masas Lacustres...", flush=True)
    join_lago = gpd.sjoin_nearest(gdf_sensors_3857, gdf_lagos[["nombre_clean", "tipo_clean", "geometry"]], distance_col="dist_lago_m", how="left")
    join_lago = join_lago.groupby(join_lago.index).first()
else:
    print("No se encontraron Masas Lacustres en el área de estudio.", flush=True)
    join_lago = pd.DataFrame(index=gdf_sensors_3857.index)
    join_lago["nombre_clean"] = "Sin masas lacustres"
    join_lago["tipo_clean"] = "N/A"
    join_lago["dist_lago_m"] = np.nan
    join_lago["index_right"] = np.nan

# Ensamblar Dataframe Consolidado de los Sensores
df_res = pd.DataFrame({
    "sensor_id": gdf_sensors_3857["sensor_id"],
    "calle_sensor": gdf_sensors_3857["calle_sensor"],
    "tipo_campana_medicion": gdf_sensors_3857["tipo_campana_medicion"],
    "latitud": gdf_sensors_3857["latitud"].round(6),
    "longitud": gdf_sensors_3857["longitud"].round(6),
    "rio_mas_cercano": join_rio["nombre_clean"],
    "tipo_rio": join_rio["tipo_clean"],
    "distancia_rio_m": join_rio["dist_rio_m"].round(2),
    "lago_mas_cercano": join_lago["nombre_clean"],
    "tipo_lago": join_lago["tipo_clean"],
    "distancia_lago_m": join_lago["dist_lago_m"].round(2),
})

# Determinar cuerpo de agua global más cercano de forma segura (considerando si falta río o lago)
def evaluar_global(r):
    d_rio = r["distancia_rio_m"]
    d_lago = r["distancia_lago_m"]
    if pd.notna(d_rio) and pd.notna(d_lago):
        if d_rio <= d_lago:
            return pd.Series([r["rio_mas_cercano"], r["tipo_rio"], d_rio])
        else:
            return pd.Series([r["lago_mas_cercano"], r["tipo_lago"], d_lago])
    elif pd.notna(d_rio):
        return pd.Series([r["rio_mas_cercano"], r["tipo_rio"], d_rio])
    elif pd.notna(d_lago):
        return pd.Series([r["lago_mas_cercano"], r["tipo_lago"], d_lago])
    else:
        return pd.Series(["Sin cuerpo de agua", "N/A", np.nan])

df_res[["cuerpo_agua_mas_cercano_global", "tipo_cuerpo_agua_global", "distancia_cuerpo_agua_global_m"]] = df_res.apply(evaluar_global, axis=1)

# Reordenar columnas según requerimiento
cols_order = [
    "sensor_id", "calle_sensor", "tipo_campana_medicion", "latitud", "longitud",
    "cuerpo_agua_mas_cercano_global", "tipo_cuerpo_agua_global", "distancia_cuerpo_agua_global_m",
    "rio_mas_cercano", "tipo_rio", "distancia_rio_m",
    "lago_mas_cercano", "tipo_lago", "distancia_lago_m"
]
df_res = df_res[cols_order].sort_values(by="sensor_id").reset_index(drop=True)
print("Cálculo finalizado exitosamente. Muestra de los primeros 5 sensores:")
print(df_res.head(5).to_string())


Calculando distancia más cercana a Ríos...


No se encontraron Masas Lacustres en el área de estudio.


Cálculo finalizado exitosamente. Muestra de los primeros 5 sensores:
  sensor_id          calle_sensor tipo_campana_medicion    latitud   longitud cuerpo_agua_mas_cercano_global tipo_cuerpo_agua_global  distancia_cuerpo_agua_global_m           rio_mas_cercano tipo_rio  distancia_rio_m     lago_mas_cercano tipo_lago  distancia_lago_m
0       C11              Mac Iver             Colectora -38.762104 -72.599915       Estero Sin Nombre / Dren                  Estero                          550.06  Estero Sin Nombre / Dren   Estero           550.06  Sin masas lacustres       N/A               NaN
1       C14              Mac Iver             Colectora -38.758926 -72.593900       Estero Sin Nombre / Dren                  Estero                           74.47  Estero Sin Nombre / Dren   Estero            74.47  Sin masas lacustres       N/A               NaN
2        C9             Manquehue             Colectora -38.763624 -72.596285       Estero Sin Nombre / Dren                  Estero 

In [5]:
# 5. Exportar Archivos CSV Separados por Comas
out_csv = DATOS_OUT / f"p4_puntos_cuerpos_agua_{NOMBRE_CIUDAD.lower()}.csv"
df_res.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"CSV principal guardado -> {out_csv.name} ({len(df_res)} registros)")

def modo_seguro(x):
    m = x.dropna().mode()
    return m.iloc[0] if not m.empty else "--"

# Resumen Estadístico agrupado por Tipo de Campaña de Medición
df_resumen_campana = df_res.groupby("tipo_campana_medicion").agg(
    total_sensores=("sensor_id", "count"),
    distancia_promedio_m=("distancia_cuerpo_agua_global_m", "mean"),
    distancia_minima_m=("distancia_cuerpo_agua_global_m", "min"),
    distancia_maxima_m=("distancia_cuerpo_agua_global_m", "max"),
    cuerpo_agua_mas_frecuente=("cuerpo_agua_mas_cercano_global", modo_seguro)
).reset_index()

df_resumen_campana["distancia_promedio_m"] = df_resumen_campana["distancia_promedio_m"].round(2)
df_resumen_campana["distancia_minima_m"] = df_resumen_campana["distancia_minima_m"].round(2)
df_resumen_campana["distancia_maxima_m"] = df_resumen_campana["distancia_maxima_m"].round(2)

out_resumen_csv = DATOS_OUT / f"p4_resumen_distancias_{NOMBRE_CIUDAD.lower()}.csv"
df_resumen_campana.to_csv(out_resumen_csv, index=False, encoding="utf-8-sig")
print(f"CSV de resumen por campaña guardado -> {out_resumen_csv.name}")
print(df_resumen_campana.to_string(index=False))


CSV principal guardado -> p4_puntos_cuerpos_agua_padrelascasas_abr2014.csv (15 registros)
CSV de resumen por campaña guardado -> p4_resumen_distancias_padrelascasas_abr2014.csv
tipo_campana_medicion  total_sensores  distancia_promedio_m  distancia_minima_m  distancia_maxima_m cuerpo_agua_mas_frecuente
            Colectora               3                273.28               74.47              550.06  Estero Sin Nombre / Dren
                Local              10                663.10              119.64             1442.56  Estero Sin Nombre / Dren
              Troncal               2               1235.75              651.67             1819.84  Estero Sin Nombre / Dren


In [6]:
# 6. Mapa Web Interactivo Folium
centro_lat = gdf_sensors_4326.geometry.y.mean()
centro_lon = gdf_sensors_4326.geometry.x.mean()

m = folium.Map(location=[centro_lat, centro_lon], zoom_start=12, tiles="CartoDB positron", control_scale=True)

# Capa de Masas Lacustres (si existen)
if len(gdf_lagos) > 0:
    fg_lagos = FeatureGroup(name=f"Masas Lacustres / Lagos {NOMBRE_CIUDAD}", show=True)
    gdf_lagos_4326 = gdf_lagos.to_crs(4326)
    for _, r in gdf_lagos_4326.iterrows():
        folium.GeoJson(
            r.geometry,
            style_function=lambda x: {"color": "#1d4ed8", "weight": 1, "fillColor": "#60a5fa", "fillOpacity": 0.45},
            tooltip=f"{r['nombre_clean']} ({r['tipo_clean']})"
        ).add_to(fg_lagos)
    fg_lagos.add_to(m)

# Capa de Red Hidrográfica (Ríos) (si existen)
if len(gdf_rios) > 0:
    fg_rios = FeatureGroup(name=f"Red Hidrográfica / Ríos {NOMBRE_CIUDAD}", show=True)
    gdf_rios_4326 = gdf_rios.to_crs(4326)
    for _, r in gdf_rios_4326.iterrows():
        w = 2.5 if r.get("Dren_Tipo") == "Río" or r.get("tipo_clean") == "Río" else 1.2
        folium.GeoJson(
            r.geometry,
            style_function=lambda x, weight=w: {"color": "#0284c7", "weight": weight, "opacity": 0.85},
            tooltip=f"{r['nombre_clean']} ({r['tipo_clean']})"
        ).add_to(fg_rios)
    fg_rios.add_to(m)

# Capa de Sensores de Medición
COLORES_CAMPANA = {
    "Troncal": "#ef4444", "Colectora": "#f97316", "Servicio": "#3b82f6",
    "Local": "#10b981", "Expresa": "#a855f7", "Estaciones": "#8b5cf6", "Otro": "#6b7280"
}

fg_sensores = FeatureGroup(name=f"Puntos de Medición ({NOMBRE_CIUDAD} - {len(df_res)} Sensores)", show=True)
for _, r in df_res.iterrows():
    c_color = COLORES_CAMPANA.get(r["tipo_campana_medicion"], "#6b7280")
    d_glob_str = f"{r['distancia_cuerpo_agua_global_m']:.2f} m" if pd.notna(r['distancia_cuerpo_agua_global_m']) else "N/D"
    d_rio_str = f"{r['distancia_rio_m']:.2f} m" if pd.notna(r['distancia_rio_m']) else "N/D"
    d_lago_str = f"{r['distancia_lago_m']:.2f} m" if pd.notna(r['distancia_lago_m']) else "N/D"
    popup_text = f"""
    <b>Sensor ID:</b> {r['sensor_id']}<br>
    <b>Calle:</b> {r['calle_sensor']}<br>
    <b>Campaña:</b> {r['tipo_campana_medicion']}<br>
    <hr style='margin:4px 0;'>
    <b>Cuerpo de Agua Más Cercano:</b> {r['cuerpo_agua_mas_cercano_global']}<br>
    <b>Tipo:</b> {r['tipo_cuerpo_agua_global']}<br>
    <b>Distancia:</b> {d_glob_str}<br>
    <hr style='margin:4px 0;'>
    <b>Río más cercano:</b> {r['rio_mas_cercano']} ({d_rio_str})<br>
    <b>Lago/Laguna más cercana:</b> {r['lago_mas_cercano']} ({d_lago_str})
    """
    folium.CircleMarker(
        location=[r["latitud"], r["longitud"]],
        radius=6,
        color="white",
        weight=1.5,
        fill=True,
        fill_color=c_color,
        fill_opacity=0.9,
        tooltip=f"Sensor {r['sensor_id']}: {r['cuerpo_agua_mas_cercano_global']} ({d_glob_str})",
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(fg_sensores)

fg_sensores.add_to(m)
LayerControl(collapsed=False).add_to(m)

mapa_out = APP_DIR / f"mapa_cuerpos_agua_{NOMBRE_CIUDAD.lower()}.html"
m.save(str(mapa_out))
print(f"Mapa web Folium guardado -> {mapa_out.name}")


Mapa web Folium guardado -> mapa_cuerpos_agua_padrelascasas_abr2014.html


In [7]:
# 7. Figura Estática Gráfica en Matplotlib
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [1.2, 1]})

# 1. Mapa Espacial de Puntos y Red Hidrográfica
if len(gdf_rios) > 0:
    gdf_rios.plot(ax=ax1, color="#0284c7", linewidth=1.0, alpha=0.7, label="Ríos / Drenes")
if len(gdf_lagos) > 0:
    gdf_lagos.plot(ax=ax1, color="#60a5fa", edgecolor="#1d4ed8", alpha=0.5, label="Masas Lacustres")

for cmp, grp in df_res.groupby("tipo_campana_medicion"):
    c = COLORES_CAMPANA.get(cmp, "#6b7280")
    pts = gdf_sensors_3857[gdf_sensors_3857["sensor_id"].isin(grp["sensor_id"])]
    pts.plot(ax=ax1, color=c, markersize=35, label=f"Sensores {cmp}", zorder=5)

ax1.set_title(f"{NOMBRE_CIUDAD} — Ubicación de Sensores y Cuerpos de Agua", fontsize=11, fontweight="bold")
ax1.set_axis_off()
ax1.legend(loc="upper left", fontsize=8, frameon=True, facecolor="white", framealpha=0.9)

# 2. Gráfico de Barras de Distancia Promedio por Tipo de Campaña
orden_campanas = ["Troncal", "Colectora", "Servicio", "Local", "Expresa", "Estaciones", "Otro"]
campanas_presentes = [c for c in orden_campanas if c in df_resumen_campana["tipo_campana_medicion"].values]
for c in df_resumen_campana["tipo_campana_medicion"].unique():
    if c not in campanas_presentes:
        campanas_presentes.append(c)

df_plot = df_resumen_campana.set_index("tipo_campana_medicion").reindex(campanas_presentes).dropna(subset=["distancia_promedio_m"]).reset_index()
colores_barras = [COLORES_CAMPANA.get(c, "#6b7280") for c in df_plot["tipo_campana_medicion"]]

if len(df_plot) > 0:
    bars = ax2.bar(df_plot["tipo_campana_medicion"], df_plot["distancia_promedio_m"], color=colores_barras, edgecolor="white", width=0.55)
    for b in bars:
        yval = b.get_height()
        ax2.text(b.get_x() + b.get_width()/2.0, yval + 15, f"{yval:.1f} m", ha='center', va='bottom', fontsize=9, fontweight='bold')
    max_dist = df_plot["distancia_promedio_m"].max()
    ax2.set_ylim(0, max(max_dist * 1.25, 10))
else:
    ax2.text(0.5, 0.5, "Sin datos de distancia calculada", ha='center', va='center', fontsize=12)
    ax2.set_ylim(0, 100)

ax2.set_title("Distancia Promedio al Cuerpo de Agua Más Cercano por Campaña", fontsize=11, fontweight="bold")
ax2.set_ylabel("Distancia Promedio (Metros)", fontsize=10)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
fig_out = DATOS_OUT / f"figura_distancia_cuerpos_agua_{NOMBRE_CIUDAD.lower()}.png"
plt.savefig(fig_out, dpi=300)
plt.close()
print(f"Figura estática guardada -> {fig_out.name}")


Figura estática guardada -> figura_distancia_cuerpos_agua_padrelascasas_abr2014.png


In [8]:
# ============================================================
# 8. Mapa Cartográfico de Conexión a Cuerpos de Agua (Ríos y Lagos)
# ============================================================
from shapely.geometry import LineString
from shapely.ops import nearest_points

print("Generando líneas de conexión euclidiana...", flush=True)

lineas_rio_list = []
lineas_lago_list = []

for idx, sensor_row in gdf_sensors_3857.iterrows():
    pt_sensor = sensor_row.geometry
    
    # Conexión hacia río más cercano (si hay río)
    if len(gdf_rios) > 0 and pd.notna(join_rio.loc[idx, "dist_rio_m"]):
        rio_idx = join_rio.loc[idx, "index_right"]
        if pd.notna(rio_idx) and rio_idx in gdf_rios.index:
            geom_rio = gdf_rios.loc[rio_idx].geometry
            _, pt_rio = nearest_points(pt_sensor, geom_rio)
            lineas_rio_list.append({"sensor_id": sensor_row["sensor_id"], "geometry": LineString([pt_sensor, pt_rio])})
            
    # Conexión hacia lago más cercano (si hay lago)
    if len(gdf_lagos) > 0 and pd.notna(join_lago.loc[idx, "dist_lago_m"]):
        lago_idx = join_lago.loc[idx, "index_right"]
        if pd.notna(lago_idx) and lago_idx in gdf_lagos.index:
            geom_lago = gdf_lagos.loc[lago_idx].geometry
            _, pt_lago = nearest_points(pt_sensor, geom_lago)
            lineas_lago_list.append({"sensor_id": sensor_row["sensor_id"], "geometry": LineString([pt_sensor, pt_lago])})

gdf_lineas_rio = gpd.GeoDataFrame(lineas_rio_list, crs=3857) if lineas_rio_list else gpd.GeoDataFrame(columns=["sensor_id", "geometry"], crs=3857)
gdf_lineas_lago = gpd.GeoDataFrame(lineas_lago_list, crs=3857) if lineas_lago_list else gpd.GeoDataFrame(columns=["sensor_id", "geometry"], crs=3857)

print(f"Líneas trazadas: {len(gdf_lineas_rio)} hacia ríos y {len(gdf_lineas_lago)} hacia lagos/lagunas.")

# 2. Configurar la Figura Espacial
fig, ax = plt.subplots(figsize=(15, 12), facecolor="#f8fafc")
ax.set_facecolor="#f1f5f9"

if len(gdf_lagos) > 0:
    gdf_lagos.plot(ax=ax, color="#bfdbfe", edgecolor="#2563eb", linewidth=1.2, alpha=0.75, label="Masas Lacustres (Lagos/Lagunas)")

if len(gdf_rios) > 0:
    gdf_rios.plot(ax=ax, color="#0284c7", linewidth=1.5, alpha=0.85, label="Red Hidrográfica (Ríos/Drenes)")

if len(gdf_lineas_lago) > 0:
    gdf_lineas_lago.plot(ax=ax, color="#7c3aed", linestyle=":", linewidth=1.3, alpha=0.8, label="Conexión ➔ Lago más cercano")

if len(gdf_lineas_rio) > 0:
    gdf_lineas_rio.plot(ax=ax, color="#0d9488", linestyle="--", linewidth=1.4, alpha=0.85, label="Conexión ➔ Río más cercano")

# Sensores clasificados por tipo de campaña
for cmp, grp in df_res.groupby("tipo_campana_medicion"):
    c = COLORES_CAMPANA.get(cmp, "#6b7280")
    pts = gdf_sensors_3857[gdf_sensors_3857["sensor_id"].isin(grp["sensor_id"])]
    pts.plot(ax=ax, color=c, edgecolor="black", linewidth=0.8, markersize=50, label=f"Sensor ({cmp})", zorder=6)

# Encuadre óptimo
bounds_list = [gdf_sensors_3857.total_bounds]
if len(gdf_lineas_rio) > 0:
    bounds_list.append(gdf_lineas_rio.total_bounds)
if len(gdf_lineas_lago) > 0:
    bounds_list.append(gdf_lineas_lago.total_bounds)

xmin = min(b[0] for b in bounds_list) - 1500
ymin = min(b[1] for b in bounds_list) - 1500
xmax = max(b[2] for b in bounds_list) + 1500
ymax = max(b[3] for b in bounds_list) + 1500

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Títulos y estilo
ax.set_title(f"{NOMBRE_CIUDAD.upper()} — Conexión Euclidiana de Sensores a Cuerpos de Agua", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Coordenadas Web Mercator X (m)", fontsize=10, color="#475569")
ax.set_ylabel("Coordenadas Web Mercator Y (m)", fontsize=10, color="#475569")
ax.grid(True, linestyle="--", alpha=0.3, color="#94a3b8")
ax.legend(loc="upper right", fontsize=9, frameon=True, facecolor="white", edgecolor="#cbd5e1", framealpha=0.95)

fig_conexiones_out = DATOS_OUT / f"mapa_conexiones_cuerpos_agua_{NOMBRE_CIUDAD.lower()}.png"
plt.savefig(fig_conexiones_out, dpi=300, bbox_inches="tight")
plt.close()

print(f"Nueva imagen de conexiones guardada -> {fig_conexiones_out.name}")


Generando líneas de conexión euclidiana...


Líneas trazadas: 15 hacia ríos y 0 hacia lagos/lagunas.


Nueva imagen de conexiones guardada -> mapa_conexiones_cuerpos_agua_padrelascasas_abr2014.png


In [9]:
# ============================================================
# 9. Generación de Figuras Individuales por Punto de Medición
# ============================================================
FIGURAS_P4_DIR = DATOS_OUT / "01_fig" / f"scan_{NOMBRE_CIUDAD.lower()}"
FIGURAS_P4_DIR.mkdir(parents=True, exist_ok=True)

print(f"Generando {len(df_res)} imágenes individuales con leyenda externa (lado derecho) en: {FIGURAS_P4_DIR}...", flush=True)

total_puntos = len(df_res)

for i, (idx, row) in enumerate(gdf_sensors_3857.iterrows(), start=1):
    pt_sensor = row.geometry
    sensor_id = row["sensor_id"]
    calle = row.get("calle_sensor", "s/n")
    campana = row.get("tipo_campana_medicion", "General")
    
    geoms_sub = [pt_sensor]
    has_rio = False
    has_lago = False
    
    # 1. Geometría y datos hacia el río más cercano
    if len(gdf_rios) > 0 and pd.notna(join_rio.loc[idx, "dist_rio_m"]):
        rio_idx = join_rio.loc[idx, "index_right"]
        if pd.notna(rio_idx) and rio_idx in gdf_rios.index:
            geom_rio = gdf_rios.loc[rio_idx].geometry
            nom_rio = join_rio.loc[idx, "nombre_clean"]
            dist_rio = join_rio.loc[idx, "dist_rio_m"]
            _, pt_rio_cercano = nearest_points(pt_sensor, geom_rio)
            line_rio = LineString([pt_sensor, pt_rio_cercano])
            geoms_sub.append(line_rio)
            has_rio = True
    
    # 2. Geometría y datos hacia el lago más cercano
    if len(gdf_lagos) > 0 and pd.notna(join_lago.loc[idx, "dist_lago_m"]):
        lago_idx = join_lago.loc[idx, "index_right"]
        if pd.notna(lago_idx) and lago_idx in gdf_lagos.index:
            geom_lago = gdf_lagos.loc[lago_idx].geometry
            nom_lago = join_lago.loc[idx, "nombre_clean"]
            dist_lago = join_lago.loc[idx, "dist_lago_m"]
            _, pt_lago_cercano = nearest_points(pt_sensor, geom_lago)
            line_lago = LineString([pt_sensor, pt_lago_cercano])
            geoms_sub.append(line_lago)
            has_lago = True
            
    # 3. Encuadre dinámico
    sub_geom = gpd.GeoSeries(geoms_sub, crs=3857)
    b = sub_geom.total_bounds
    dx = max(b[2] - b[0], 600)
    dy = max(b[3] - b[1], 600)
    padx = dx * 0.18
    pady = dy * 0.18
    xlim = (b[0] - padx, b[2] + padx)
    ylim = (b[1] - pady, b[3] + pady)
    
    # 4. Crear figura
    fig, ax = plt.subplots(figsize=(11, 8.5), facecolor="#f8fafc")
    ax.set_facecolor("#f1f5f9")
    
    if len(gdf_lagos) > 0:
        gdf_lagos.plot(ax=ax, color="#bfdbfe", edgecolor="#2563eb", linewidth=1.5, alpha=0.75, label="Masas Lacustres")
    if len(gdf_rios) > 0:
        gdf_rios.plot(ax=ax, color="#0284c7", linewidth=2.0, alpha=0.85, label="Red Hidrográfica")
        
    if has_lago:
        gpd.GeoSeries([line_lago], crs=3857).plot(ax=ax, color="#7c3aed", linestyle=":", linewidth=2.5, label=f"Lago: {nom_lago}\n({dist_lago:.1f} m)")
        ax.scatter([pt_lago_cercano.x], [pt_lago_cercano.y], color="#7c3aed", s=90, marker="X", zorder=6, label="Punto contacto Lago")
        
    if has_rio:
        gpd.GeoSeries([line_rio], crs=3857).plot(ax=ax, color="#0d9488", linestyle="--", linewidth=2.5, label=f"Río: {nom_rio}\n({dist_rio:.1f} m)")
        ax.scatter([pt_rio_cercano.x], [pt_rio_cercano.y], color="#0d9488", s=90, marker="X", zorder=6, label="Punto contacto Río")
        
    # Punto del sensor
    c_sensor = COLORES_CAMPANA.get(campana, "#ef4444")
    ax.scatter([pt_sensor.x], [pt_sensor.y], color=c_sensor, edgecolor="black", s=140, linewidth=1.5, zorder=7, label=f"Sensor {sensor_id} ({campana})")
    
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    
    ax.set_title(f"{NOMBRE_CIUDAD.upper()} — Sensor {sensor_id}: Distancia a Cuerpos de Agua\nCalle: {calle}", fontsize=12, fontweight="bold", pad=12)
    ax.set_xlabel("Coordenadas Web Mercator X (m)", fontsize=9, color="#475569")
    ax.set_ylabel("Coordenadas Web Mercator Y (m)", fontsize=9, color="#475569")
    ax.grid(True, linestyle="--", alpha=0.3, color="#94a3b8")
    
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        fontsize=9,
        frameon=True,
        facecolor="white",
        edgecolor="#cbd5e1",
        framealpha=0.95
    )
    
    out_file = FIGURAS_P4_DIR / f"{NOMBRE_CIUDAD.lower()}_{sensor_id}_cuerpos_agua.png"
    plt.savefig(out_file, dpi=200, bbox_inches="tight")
    plt.close(fig)
    
    if i % 10 == 0 or i == total_puntos:
        print(f"[{i}/{total_puntos}] Figuras generadas...", flush=True)

print(f"✔ Proceso finalizado. Se actualizaron {total_puntos} figuras individuales con leyenda exterior en: {FIGURAS_P4_DIR}/")


Generando 15 imágenes individuales con leyenda externa (lado derecho) en: 04_agu/01_fig/scan_padrelascasas_abr2014...


[10/15] Figuras generadas...


[15/15] Figuras generadas...


✔ Proceso finalizado. Se actualizaron 15 figuras individuales con leyenda exterior en: 04_agu/01_fig/scan_padrelascasas_abr2014/
